# einops-rearrange composite — cx12: rearrange to put reducible axes last, then reduce them

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-rearrange`, `einops-reduce`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-rearrange"
DD_ATOM_IDS = ["einops-rearrange", "einops-reduce"]
DD_SUBTOPICS = ["Einops: Rearrange", "Einops: Reduce"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Real-world tensors don't always come pre-arranged in the order you want to reduce over. `einops.rearrange` is the pure axis-reorder primitive — no shape change, just a permutation of named axes. Once the axes you want to collapse are in a convenient position, `einops.reduce` with a per-axis pattern is trivial.

Composing them means: write a rearrange pattern that ONLY permutes axes (no flattening, no size change), then write a reduce pattern that drops the right ones. The composition is the ARENA-standard pre-process-then-reduce idiom.

### Composite Exercise — rearrange to put reducible axes last, then reduce them

**Atoms exercised together**: `einops-rearrange`, `einops-reduce`

Implement `cx12_rearrange_then_reduce_max(x)` that takes a 4-D tensor of shape `(B, H, W, C)` (channels-last) and returns a `(B, C)` tensor of the per-(batch, channel) MAX over spatial H, W axes.

Two-step composition:

1. **Rearrange** the channels-last layout to channels-first via `einops.rearrange(x, 'b h w c -> b c h w')`. This is the identity-up-to-permutation atom — no axis flattening, no size change, just a permutation.
2. **Reduce** the spatial axes with `einops.reduce(permuted, 'b c h w -> b c', 'max')`. Per-(batch, channel) max over H,W.

Assert inside the fn that `permuted.shape == (B, C, H, W)` so atom 1 is visible. The point is NOT to write `x.permute(0, 3, 1, 2).amax(dim=(-2, -1))` — the point is to write both einops calls and verify each is type-pure (rearrange = pure axis reorder; reduce = pure axis drop).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx12_rearrange_then_reduce_max(x):
    raise NotImplementedError

def _test_cx12():
    x = t.arange(2 * 3 * 4 * 5).reshape(2, 3, 4, 5).float()  # (B=2, H=3, W=4, C=5)
    out = cx12_rearrange_then_reduce_max(x)
    assert out.shape == (2, 5), f'expected (2,5), got {out.shape}'
    # Cross-check vs torch: amax over H,W of the channels-first view.
    ref = x.permute(0, 3, 1, 2).amax(dim=(-2, -1))
    assert t.allclose(out, ref), f'mismatch: {out} vs {ref}'

    # Case B: random + larger spatial.
    x2 = t.randn(4, 8, 8, 3)
    out2 = cx12_rearrange_then_reduce_max(x2)
    assert out2.shape == (4, 3)
    assert t.allclose(out2, x2.permute(0, 3, 1, 2).amax(dim=(-2, -1)))

    # Case C: degenerate H=W=1 — max equals the single value per (B, C).
    x3 = t.randn(2, 1, 1, 6)
    out3 = cx12_rearrange_then_reduce_max(x3)
    assert out3.shape == (2, 6)
    assert t.allclose(out3, x3.squeeze(1).squeeze(1))
    _dd_passed.add('cx12')

_test_cx12()

<details><summary>Show solution — cx12</summary>

```python
def cx12_rearrange_then_reduce_max(x):
    B, H, W, C = x.shape
    # Atom 1: rearrange — pure axis permutation, no shape change beyond the reorder.
    permuted = rearrange(x, 'b h w c -> b c h w')
    assert permuted.shape == (B, C, H, W), permuted.shape
    # Atom 2: reduce — drop the spatial axes with 'max'.
    return reduce(permuted, 'b c h w -> b c', 'max')
```

The split is intentional: each einops call is type-pure. The rearrange does ONLY axis reordering (no parens-flatten, no kwarg-bound new axis); the reduce does ONLY axis dropping (no axis composition, no new axes). Composing them gives you the pre-process-then-reduce idiom without leaving einops vocabulary.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx12'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx12',
        'subtopics': ["Einops: Rearrange", "Einops: Reduce"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()